In [101]:
# %% [markdown]
"""
# Pipeline de Clasificación para Emisores Hα en IGAPS + AllWISE
**Objetivo**: Optimizar identificación de PNe, SySt, YSOs y otros emisores mediante fine-tuning de UMAP + HDBSCAN
"""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import RobustScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import silhouette_score, davies_bouldin_score, silhouette_samples
from sklearn.ensemble import IsolationForest
from itertools import product
import umap
import umap.umap_ as umap
import hdbscan
from tqdm import tqdm
import joblib
import traceback
import warnings
from astropy.coordinates import SkyCoord
import astropy.units as u
from astroquery.simbad import Simbad
import plotly.express as px

In [102]:
# Configuración global
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.2)
pd.set_option('display.max_columns', 30)
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [103]:
# Concatenate all DataFrames into a single DataFrame
df = pd.read_csv("../IGAPs-emitters-wise.csv")
print(f"Datos originales cargados: {df.shape[0]} objetos, {df.shape[1]} columnas")

Datos originales cargados: 9814 objetos, 205 columnas


In [104]:
# %% [markdown]
"""
### Aplicar Filtrado de Calidad (Versión Mejorada)
"""
# %%
def apply_quality_filters(df):
    """Aplica filtros de calidad para IGAPS y AllWISE"""
    print("Aplicando filtros de calidad...")
    
    # 1. Umbrales de error fotométrico
    max_err_optical = 0.2
    max_err_Halpha = 0.3
    max_err_rU = 0.3
    max_err_wise_w1w2 = 0.3
    max_err_wise_w3w4 = 2.0

    # Máscara de errores fotométricos
    m_err = (
        (df["e_gmag"] <= max_err_optical) &
        (df["e_imag"] <= max_err_optical) &
        (df["e_Hamag"] <= max_err_Halpha) &
        (df["e_rUmag"] <= max_err_rU) &
        (df["e_rImag"] <= max_err_optical) &
        (df["e_Umag"] <= max_err_optical) &
        (df["e_W1mag"] <= max_err_wise_w1w2) &
        (df["e_W2mag"] <= max_err_wise_w1w2) &
        (df["e_W3mag"] <= max_err_wise_w3w4) &
        (df["e_W4mag"] <= max_err_wise_w3w4)
    )

    # 2. Flags de calidad IGAPS
    bands = ['g', 'rI', 'rU', 'i', 'Ha']
    m_quality_igaps = True
    for band in bands:
        # Flags de saturación
        sat_col = f'Saturated{band}'
        if sat_col in df.columns:
            df[sat_col] = df[sat_col].fillna(1)  # Tratar NaN como malos
            m_quality_igaps &= (df[sat_col] == 0)
        
        # Otros flags importantes
        for flag in ['BadPix', 'Trail', 'Truncated']:
            flag_col = f'{flag}{band}'
            if flag_col in df.columns:
                df[flag_col] = df[flag_col].fillna(1)
                m_quality_igaps &= (df[flag_col] == 0)

    # Combinar máscaras base
    mask_total = m_err & m_quality_igaps

    # 3. Flags de calidad AllWISE
    m_wise_quality = True
    wise_quality_applied = False
    
    
    # Calidad fotométrica (qph)
    if 'qph' in df.columns:
        df['qph'] = df['qph'].fillna('XXXX').astype(str)
        m_wise_quality &= df['qph'].str[0].isin(['A','B'])  # W1
        m_wise_quality &= df['qph'].str[1].isin(['A','B'])  # W2
        wise_quality_applied = True
    
    # Aplicar si hay columnas relevantes
    if wise_quality_applied:
        mask_total &= m_wise_quality
        print("✅ Filtros AllWISE aplicados")
    else:
        print("⚠️ No se encontraron flags AllWISE - Usando solo filtros IGAPS")
    
    return df[mask_total].copy()

# %%
df_filtrado = apply_quality_filters(df)
print(f"Datos después de filtrado de calidad: {df_filtrado.shape[0]} objetos")


Aplicando filtros de calidad...
✅ Filtros AllWISE aplicados
Datos después de filtrado de calidad: 2742 objetos


In [109]:
# %% [markdown]
"""
## Paso 2: Ingeniería de Características Robusta
"""
# %%
def create_features(df):
    """Crea características fotométricas con manejo robusto de NaN"""
    features = pd.DataFrame(index=df.index)
    
    # Función segura para cálculo de colores
    def safe_color(mag1, mag2, default=np.nan):
        if mag1 in df.columns and mag2 in df.columns:
            # Manejar casos donde una magnitud es NaN
            valid_mask = df[mag1].notna() & df[mag2].notna()
            return np.where(valid_mask, df[mag1] - df[mag2], default)
        return np.full(len(df), default)
    
    # ---- Colores ópticos clave ----
    features['U_g'] = safe_color('Umag', 'gmag')  # ¡Ahora incluido!
    features['g_r'] = safe_color('gmag', 'rImag')
    features['r_i'] = safe_color('rImag', 'imag')
    features['r_Ha'] = safe_color('rImag', 'Hamag')  # Exceso Hα principal
    features['i_Ha'] = safe_color('imag', 'Hamag')   # Alternativo para objetos rojos
    
    # ---- Variabilidad ----
    if 'rImag' in df.columns and 'rUmag' in df.columns:
        # Usar diferencia absoluta para la variabilidad
        features['var_r'] = np.abs(df['rImag'] - df['rUmag'])
    
    # ---- Colores Óptico-IR ----
    features['g_W1'] = safe_color('gmag', 'W1mag')
    features['r_W2'] = safe_color('rImag', 'W2mag')
    features['Ha_W1'] = safe_color('Hamag', 'W1mag')
    features['Ha_W2'] = safe_color('Hamag', 'W2mag')
    features['i_K'] = safe_color('imag', 'Kmag')     # Si está disponible
    features['J_r'] = safe_color('Jmag', 'rImag')    # Si está disponible
    
    # ---- Colores IR-IR ----
    features['W1_W2'] = safe_color('W1mag', 'W2mag')
    features['J_H'] = safe_color('Jmag', 'Hmag')      # Si está disponible
    features['H_K'] = safe_color('Hmag', 'Kmag')      # Si está disponible
    features['W1_J'] = safe_color('W1mag', 'Jmag')    # Si está disponible
    
    # ---- Nuevos colores con W3/W4 ----
    features['W3_W4'] = safe_color('W3mag', 'W4mag')   # Polvo frío (crítico!)
    features['W2_W4'] = safe_color('W2mag', 'W4mag')   # Rango amplio de polvo
    features['r_W4'] = safe_color('rImag', 'W4mag')    # Exceso IR extremo
    
    # ---- Manejo de valores faltantes ----
    # 1. Imputación para características críticas
    critical_features = ['r_Ha', 'g_r', 'W3_W4', 'var_r', 'U_g']
    for feat in critical_features:
        if feat in features.columns:
            # Imputar con la mediana solo si hay valores faltantes
            if features[feat].isna().any():
                median_val = features[feat].median()
                features[feat] = features[feat].fillna(median_val)
    
    # 2. Eliminar filas con >50% de features faltantes
    features = features.dropna(thresh=len(features.columns)//2)
    
    # 3. Eliminar columnas con >30% de valores faltantes
    missing_cols = features.columns[features.isna().mean() > 0.3]
    features = features.drop(columns=missing_cols)
    
    return features

In [110]:
# %%
print("Creando características...")
features_df = create_features(df_filtrado)
print(f"Características creadas: {features_df.shape[0]} objetos, {features_df.shape[1]} características")
print("Características:", features_df.columns.tolist())

Creando características...
Características creadas: 2742 objetos, 19 características
Características: ['U_g', 'g_r', 'r_i', 'r_Ha', 'i_Ha', 'var_r', 'g_W1', 'r_W2', 'Ha_W1', 'Ha_W2', 'i_K', 'J_r', 'W1_W2', 'J_H', 'H_K', 'W1_J', 'W3_W4', 'W2_W4', 'r_W4']


In [112]:
# %% [markdown]
"""
## Paso 3: Preprocesamiento Mejorado
"""
# %%
def robust_preprocessing(features_df):
    """Pipeline de preprocesamiento con manejo robusto de NaN"""
    # 1. Imputación multivariada avanzada
    imputer = IterativeImputer(
        estimator=RandomForestRegressor(n_estimators=50, random_state=RANDOM_STATE),
        max_iter=15,
        random_state=RANDOM_STATE,
        verbose=0
    )
    X_imputed = imputer.fit_transform(features_df)
    
    # 2. Detección de outliers
    clf = IsolationForest(contamination=0.05, random_state=RANDOM_STATE)
    outliers = clf.fit_predict(X_imputed)
    inliers = outliers == 1
    
    # 3. Escalado robusto
    scaler = RobustScaler(quantile_range=(10, 90))
    X_scaled = scaler.fit_transform(X_imputed[inliers])
    
    return X_scaled, inliers, imputer, scaler

In [113]:
# %%
print("Preprocesando datos...")
X, inliers_mask, imputer, scaler = robust_preprocessing(features_df)
df_clean = df_filtrado.loc[features_df.index[inliers_mask]].copy()
features_clean = features_df.loc[inliers_mask]

print(f"Datos limpios para clustering: {X.shape[0]} objetos")

Preprocesando datos...
Datos limpios para clustering: 2604 objetos


In [115]:
#% [markdown]
"""
## Paso 4: Configuración de Experimentos
"""
# %%
# Métodos de reducción dimensional
DR_METHODS = ['UMAP', 'PCA', 'TSNE']

# Métricas para UMAP/TSNE (PCA siempre usa euclidiana)
METRICS = ['euclidean', 'manhattan', 'cosine', 'correlation']

# Parámetros de HDBSCAN para optimizar
HDBSCAN_PARAMS = {
    'min_cluster_size': [15, 30, 50],
    'min_samples': [5, 10, 15],
    'cluster_selection_epsilon': [0.3, 0.5, 0.7]
}

# Almacenar resultados
results = []

In [116]:
# Función de evaluación mejorada
def evaluate_clustering(embedding, clusters):
    """Calcula métricas de calidad de clustering"""
    if len(np.unique(clusters)) < 2:
        return {'silhouette': -1, 'davies_bouldin': 100}
    
    try:
        return {
            'silhouette': silhouette_score(embedding, clusters),
            'davies_bouldin': davies_bouldin_score(embedding, clusters)
        }
    except:
        return {'silhouette': -1, 'davies_bouldin': 100}


In [117]:
# %% [markdown]
"""
## Paso 5: Búsqueda de Hiperparámetros (Optimizada)
"""
# %%
def run_experiment(args):
    """Función para ejecutar un experimento individual (para paralelización)"""
    dr_method, metric, min_size, min_samp, eps, X = args
    
    try:
        # Reducción dimensional
        if dr_method == 'UMAP':
            reducer = umap.UMAP(
                n_neighbors=min(30, len(X)//10),  # Ajuste automático para tamaño de muestra
                min_dist=0.1,
                n_components=3,
                metric=metric,
                random_state=RANDOM_STATE,
                low_memory=False
            )
            embedding = reducer.fit_transform(X)
            
        elif dr_method == 'PCA':
            reducer = PCA(n_components=0.95, random_state=RANDOM_STATE)
            embedding = reducer.fit_transform(X)
            
        elif dr_method == 'TSNE':
            # Muestra representativa para t-SNE (más eficiente)
            if len(X) > 5000:
                idx_sample = np.random.choice(len(X), 5000, replace=False)
                X_sample = X[idx_sample]
            else:
                X_sample = X
                
            reducer = TSNE(
                n_components=3,
                perplexity=min(40, len(X_sample)//100),  # Ajuste automático
                metric=metric,
                random_state=RANDOM_STATE,
                init='pca',
                method='barnes_hut' if len(X_sample) > 1000 else 'exact'
            )
            embedding_sample = reducer.fit_transform(X_sample)
            
            # Para el clustering completo usamos aproximación
            if len(X) > 5000:
                knn = KNeighborsRegressor(n_neighbors=10)
                knn.fit(X_sample, embedding_sample)
                embedding = knn.predict(X)
            else:
                embedding = embedding_sample
        
        # Clustering HDBSCAN
        clusterer = hdbscan.HDBSCAN(
            min_cluster_size=min_size,
            min_samples=min_samp,
            cluster_selection_epsilon=eps,
            gen_min_span_tree=True,
            core_dist_n_jobs=1  # Usar un solo núcleo para evitar conflictos
        )
        clusters = clusterer.fit_predict(embedding)
        
        # Evaluación (solo si hay clusters válidos)
        if len(np.unique(clusters)) > 1:
            metrics = evaluate_clustering(embedding, clusters)
            n_clusters = len(np.unique(clusters[clusters != -1]))
        else:
            metrics = {'silhouette': -1, 'davies_bouldin': 100}
            n_clusters = 0
            
        return {
            'dr_method': dr_method,
            'metric': metric,
            'min_cluster_size': min_size,
            'min_samples': min_samp,
            'epsilon': eps,
            'n_clusters': n_clusters,
            'silhouette': metrics['silhouette'],
            'davies_bouldin': metrics['davies_bouldin'],
            'noise_ratio': np.mean(clusters == -1) if len(clusters) > 0 else 1.0
        }
        
    except Exception as e:
        print(f"Error en {dr_method}/{metric}: {str(e)}")
        return None

In [118]:
# %%
print("Configurando experimentos...")
experiments = []
for dr_method, metric in product(DR_METHODS, METRICS):
    # Filtro de compatibilidad
    if dr_method == 'PCA' and metric != 'euclidean':
        continue
    if dr_method == 'TSNE' and metric not in ['euclidean', 'manhattan', 'cosine']:
        continue
        
    for min_size, min_samp, eps in product(
        HDBSCAN_PARAMS['min_cluster_size'],
        HDBSCAN_PARAMS['min_samples'],
        HDBSCAN_PARAMS['cluster_selection_epsilon']
    ):
        experiments.append((dr_method, metric, min_size, min_samp, eps, X))

print(f"Total de experimentos configurados: {len(experiments)}")


Configurando experimentos...
Total de experimentos configurados: 216


In [119]:
# Ejecutar en paralelo
print("Iniciando búsqueda de hiperparámetros...")
with joblib.parallel_backend('loky', n_jobs=-1):
    results = joblib.Parallel(verbose=10)(
        joblib.delayed(run_experiment)(exp) for exp in experiments
    )

Iniciando búsqueda de hiperparámetros...


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 12 concurrent workers.
2025-06-19 17:11:01.925670: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-06-19 17:11:01.925675: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-06-19 17:11:01.925677: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-06-19 17:11:01.925675: I tensorflow/

2025-06-19 17:11:04.283855: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-06-19 17:11:04.283874: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-06-19 17:11:04.283871: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025

/home/luisangel/.conda/envs/luis_env_v2/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/luisangel/.conda/envs/luis_env_v2/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/luisangel/.conda/envs/luis_env_v2/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/luisangel/.conda/envs/luis_env_v2/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/luisangel/.conda/envs/luis_env_v2/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: F

[Parallel(n_jobs=-1)]: Done   8 tasks      | elapsed:   57.5s
/home/luisangel/.conda/envs/luis_env_v2/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/luisangel/.conda/envs/luis_env_v2/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/home/luisangel/.conda/envs/luis_env_v2/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/luisangel/.conda/envs/luis_env_v2/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/home/luisangel/.conda/envs/luis_env_v2/lib/python3.12/site-packages/sklearn/utils/deprecation.py:15

...

/home/luisangel/.conda/envs/luis_env_v2/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/luisangel/.conda/envs/luis_env_v2/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/luisangel/.conda/envs/luis_env_v2/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/luisangel/.conda/envs/luis_env_v2/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/luisangel/.conda/envs/luis_env_v2/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: F

/home/luisangel/.conda/envs/luis_env_v2/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/luisangel/.conda/envs/luis_env_v2/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/luisangel/.conda/envs/luis_env_v2/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/luisangel/.conda/envs/luis_env_v2/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/luisangel/.conda/envs/luis_env_v2/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: F

/home/luisangel/.conda/envs/luis_env_v2/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/luisangel/.conda/envs/luis_env_v2/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/luisangel/.conda/envs/luis_env_v2/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/luisangel/.conda/envs/luis_env_v2/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/luisangel/.conda/envs/luis_env_v2/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: F

/home/luisangel/.conda/envs/luis_env_v2/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/luisangel/.conda/envs/luis_env_v2/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/luisangel/.conda/envs/luis_env_v2/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/luisangel/.conda/envs/luis_env_v2/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/luisangel/.conda/envs/luis_env_v2/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: F

/home/luisangel/.conda/envs/luis_env_v2/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/luisangel/.conda/envs/luis_env_v2/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/luisangel/.conda/envs/luis_env_v2/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/luisangel/.conda/envs/luis_env_v2/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/luisangel/.conda/envs/luis_env_v2/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: F

In [120]:
# Filtrar resultados fallidos
results = [res for res in results if res is not None]
results_df = pd.DataFrame(results)
results_df.to_csv('../clustering_tuning_results.csv', index=False)
print("Búsqueda completada. Resultados guardados.")

Búsqueda completada. Resultados guardados.


In [121]:
# %% [markdown]
"""
## Paso 6: Selección del Mejor Modelo
"""
# %%
# Encontrar la mejor configuración
if not results_df.empty:
    # Filtrar configuraciones con clusters válidos
    valid_results = results_df[results_df['n_clusters'] > 1]
    
    if not valid_results.empty:
        # Mejor por silhouette score
        best_run = valid_results.loc[valid_results['silhouette'].idxmax()]
    else:
        # Fallback a la primera configuración si no hay válidas
        best_run = results_df.iloc[0]
else:
    raise RuntimeError("Todos los experimentos fallaron. Verificar datos de entrada.")

print("\nMejor configuración encontrada:")
print(best_run)



Mejor configuración encontrada:
dr_method               UMAP
metric                cosine
min_cluster_size          15
min_samples               15
epsilon                  0.7
n_clusters                 2
silhouette          0.254141
davies_bouldin       1.78277
noise_ratio         0.050691
Name: 62, dtype: object


In [122]:
# %% [markdown]
"""
## Paso 7: Entrenamiento del Modelo Final
"""
# %%
# Configuración óptima
FINAL_DR = best_run['dr_method']
FINAL_METRIC = best_run['metric']
FINAL_MIN_SIZE = int(best_run['min_cluster_size'])
FINAL_MIN_SAMPLES = int(best_run['min_samples'])
FINAL_EPS = best_run['epsilon']

print(f"\nEntrenando modelo final con: {FINAL_DR} + HDBSCAN")


Entrenando modelo final con: UMAP + HDBSCAN


In [123]:
# Reducción dimensional final
if FINAL_DR == 'UMAP':
    reducer_final = umap.UMAP(
        n_neighbors=min(30, len(X)//10),
        min_dist=0.1,
        n_components=3,
        metric=FINAL_METRIC,
        random_state=RANDOM_STATE,
        low_memory=False
    )
    embedding_final = reducer_final.fit_transform(X)
    
elif FINAL_DR == 'PCA':
    reducer_final = PCA(n_components=0.95, random_state=RANDOM_STATE)
    embedding_final = reducer_final.fit_transform(X)
    
elif FINAL_DR == 'TSNE':
    # Muestra representativa para t-SNE
    if len(X) > 5000:
        idx_sample = np.random.choice(len(X), 5000, replace=False)
        X_sample = X[idx_sample]
        reducer_final = TSNE(
            n_components=3,
            perplexity=min(40, len(X_sample)//100),
            metric=FINAL_METRIC,
            random_state=RANDOM_STATE,
            init='pca',
            method='barnes_hut'
        )
        embedding_sample = reducer_final.fit_transform(X_sample)
        
        # Extrapolar a todo el dataset
        knn = KNeighborsRegressor(n_neighbors=10)
        knn.fit(X_sample, embedding_sample)
        embedding_final = knn.predict(X)
    else:
        reducer_final = TSNE(
            n_components=3,
            perplexity=min(40, len(X)//100),
            metric=FINAL_METRIC,
            random_state=RANDOM_STATE,
            init='pca'
        )
        embedding_final = reducer_final.fit_transform(X)

# Clustering final
clusterer_final = hdbscan.HDBSCAN(
    min_cluster_size=FINAL_MIN_SIZE,
    min_samples=FINAL_MIN_SAMPLES,
    cluster_selection_epsilon=FINAL_EPS,
    gen_min_span_tree=True,
    core_dist_n_jobs=-1  # Usar todos los núcleos
)
clusters_final = clusterer_final.fit_predict(embedding_final)

# Añadir resultados al DataFrame
df_clean['cluster'] = clusters_final
df_clean['UMAP1'] = embedding_final[:, 0]
df_clean['UMAP2'] = embedding_final[:, 1]
if embedding_final.shape[1] > 2:
    df_clean['UMAP3'] = embedding_final[:, 2]


ValueError: Epsilon must be a float value greater than or equal to 0!